# Speedup analysis

Timed operation: `RandomForestClassifier.fit(train_df)` on `data/final_preprocessed` (same as [`data-modeling.ipynb`](./data-modeling.ipynb)).

In [4]:
import time

PARQUET_PATH = "data/final_preprocessed"
N_BASE = 1
N_FULL = 7
SEED = 42
NUM_TREES = 20
MAX_DEPTH = 10

# fill in after timing runs (seconds)
T_1 = 7593.41
T_7 = 1025.67

## Baseline: 1 executor

In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

spark = SparkSession.builder \
    .appName("speedup-1-exec") \
    .config("spark.executor.instances", "1") \
    .config("spark.executor.memory", "126g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

df = spark.read.parquet(PARQUET_PATH)
ml_df = df.select(F.col("EDUC").cast("double").alias("label"), "REALINCTOT_Z", "AGE_Z", "STATE_OH", "SEX_OH", "RACE_OH")
assembler = VectorAssembler(inputCols=["REALINCTOT_Z", "AGE_Z", "STATE_OH", "SEX_OH", "RACE_OH"], outputCol="features")
ml_df = assembler.transform(ml_df)
train_df, val_df, test_df = ml_df.randomSplit([0.70, 0.15, 0.15], seed=SEED)
rf = RandomForestClassifier(labelCol="label", featuresCol="features", predictionCol="prediction", numTrees=NUM_TREES, maxDepth=MAX_DEPTH, seed=SEED)

times = []
for i in range(3):
    start = time.time()
    rf.fit(train_df)
    elapsed = time.time() - start
    times.append(elapsed)
    tag = "(warmup - discard)" if i == 0 else ""
    print(f"run {i+1}: {elapsed:.2f} sec {tag}")

T_1 = sum(times[1:]) / len(times[1:])
print(f"T_1 (avg runs 2-3) = {T_1:.2f} sec")

run 1: 6407.61 sec (warmup - discard)
run 2: 7411.36 sec 
run 3: 7775.46 sec 
T_1 (avg runs 2-3) = 7593.41 sec


In [3]:
spark.stop()

## Scaled: 7 executors

In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

spark = SparkSession.builder \
    .appName("speedup-7-exec") \
    .config("spark.executor.instances", "7") \
    .config("spark.executor.memory", "18g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

df = spark.read.parquet(PARQUET_PATH)
ml_df = df.select(F.col("EDUC").cast("double").alias("label"), "REALINCTOT_Z", "AGE_Z", "STATE_OH", "SEX_OH", "RACE_OH")
assembler = VectorAssembler(inputCols=["REALINCTOT_Z", "AGE_Z", "STATE_OH", "SEX_OH", "RACE_OH"], outputCol="features")
ml_df = assembler.transform(ml_df)
train_df, val_df, test_df = ml_df.randomSplit([0.70, 0.15, 0.15], seed=SEED)
rf = RandomForestClassifier(labelCol="label", featuresCol="features", predictionCol="prediction", numTrees=NUM_TREES, maxDepth=MAX_DEPTH, seed=SEED)

times = []
for i in range(3):
    start = time.time()
    rf.fit(train_df)
    elapsed = time.time() - start
    times.append(elapsed)
    tag = "(warmup - discard)" if i == 0 else ""
    print(f"run {i+1}: {elapsed:.2f} sec {tag}")

T_7 = sum(times[1:]) / len(times[1:])
print(f"T_7 (avg runs 2-3) = {T_7:.2f} sec")

run 1: 1043.03 sec (warmup - discard)
run 2: 1014.34 sec 
run 3: 1037.00 sec 
T_7 (avg runs 2-3) = 1025.67 sec


In [3]:
spark.stop()

## Speedup, efficiency, Amdahl

Using `T_1` and `T_7` in the constants cell we run:

In [ ]:
if T_1 is None or T_7 is None:
    raise ValueError("Set T_1 and T_7 (seconds) in the constants cell first")

speedup = T_1 / T_7
efficiency = speedup / N_FULL
p = (N_FULL * (speedup - 1)) / (speedup * (N_FULL - 1))
amdahl_at_n = 1 / ((1 - p) + p / N_FULL)

print(f"T_1 = {T_1:.2f} sec")
print(f"T_7 = {T_7:.2f} sec")
print(f"speedup = {speedup:.2f}x")
print(f"efficiency = {efficiency:.1%}")
print(f"parallelizable fraction p = {p:.1%}")
print(f"Amdahl theoretical speedup at n={N_FULL} (using measured p) = {amdahl_at_n:.2f}x")
print(f"achieved / theoretical = {speedup / amdahl_at_n:.1%}")

T_1 = 7593.41 sec
T_7 = 1025.67 sec
speedup = 7.40x
efficiency = 105.8%
parallelizable fraction p = 100.9%
Amdahl theoretical speedup at n=7 (using measured p) = 7.40x
achieved / theoretical = 100.0%


In [6]:
print("| Executors | Time (sec) | Speedup | Efficiency |")
print("|---|---|---|---|")
print(f"| 1 | {T_1:.1f} | 1.00x | 100% |")
print(f"| {N_FULL} | {T_7:.1f} | {speedup:.2f}x | {efficiency:.1%} |")

| Executors | Time (sec) | Speedup | Efficiency |
|---|---|---|---|
| 1 | 7593.4 | 1.00x | 100% |
| 7 | 1025.7 | 7.40x | 105.8% |
